# Using MCP Tools with Pydantic

In this notebook, we will explore how to create a reasoning action agent using tools exposed by an MCP server with Pydantic.

## Recommended Hardware

This notebook can run on the following hardware or remote resources.

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://notebooks.amd.com/github/AMDResearch/aup-ai-tutorials/blob/main/ai-agents/02-b-mcp-pydantic.ipynb)  

## Software Environment

Install ROCm on your system.

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Create a Pydantic AI agent connected to a local LLM via Ollama.
- Connect the agent to an MCP server for tool access.
- Understand how the agent uses tools to answer questions it cannot answer from its training data alone.

### Install Dependencies

Install the package dependencies needed for this notebook or series of notebooks.

First, get the `aup_config.py` script locally if needed. Then install the dependencies (`aup_setup()`). This step may take a few minutes and only needs to be done once.

In [1]:
![ -f aup_config.py ] || wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/main/ai-agents/aup_config.py

--2026-10-04 14:18:09--  https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/main/ai-agents/aup_config.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6481 (6.3K) [text/plain]
Saving to: ‘aup_config.py’

aup_config.py       100%[===================>]   6.33K  --.-KB/s    in 0s      

2026-10-04 14:18:09 (191 MB/s) - ‘aup_config.py’ saved [6481/6481]



In [2]:
from aup_config import aup_setup
aup_setup()

INFO:root:AMD Developer Cloud detected: True.
INFO:root:AUP Learning Cloud detected: False.
INFO:root:Zstd installed successfully.
INFO:root:graphviz installed successfully.
INFO:root:Pip upgraded installed successfully.
INFO:root:Pip packages installed successfully.
INFO:root:Files open_weather.py, math_server.py downloaded successfully.
INFO:root:Ollama installed successfully.
INFO:root:Ollama started in the background
INFO:root:Ollama is pulling models, this may take a while...
INFO:root:Ollama models qwen3.5:9b pulled successfully.


['qwen3.5:9b']

## Enhance the LLM with Tools

In [3]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport

from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

### Create Model Object

Use `OpenAIChatModel` to connect to the Ollama local endpoint. 

In [4]:
provider = OpenAIProvider(
    base_url='http://localhost:11434/v1',
    api_key='abc-123',
)

agent_model = OpenAIChatModel(model_name='qwen3.5:9b', provider=provider)

### Create a Pydantic Agent

Let us create the Agent object

In [5]:
agent = Agent(
    model=agent_model
)

### Create Async Agent

Let us start by creating a simple agent with no tools. We will add MCP server support in a later step.

In [6]:
async def run_agent(prompt: str) -> str:
    async with agent:
        result = await agent.run(prompt)
        return result.output

Let us ask a simple question

In [7]:
await run_agent("What is the capital of France?")

'The capital of France is **Paris**.'

Now, let us ask a question that the model cannot answer without access to tools

In [8]:
await run_agent("What time is it?")

"I can't tell you the exact current time since I don't have real-time access to clocks or calendars on your device. However, you can check the time by looking at your phone, computer, or any nearby clock! 😊 If there's anything else you'd like help with, just ask!"

### Run an MCP Server

Now, we can run an MCP server that enables us to get information about the current date and time.

In [9]:
time_server = MCPToolset(
    StdioTransport(
        command="python",
        args=[
            "-m", "mcp_server_time",
            "--local-timezone=Europe/Dublin",
        ],
    )
)

Let us update the Agent object to include the tools exposed by this MCP server. Note that we are manually providing the system prompt.

In [10]:
agent = Agent(
    model=agent_model,
    toolsets=[time_server],
    system_prompt = (
        "You are a helpful agent and you have access to this tool:\n"
        "   get_current_time(params: dict)\n"
        "When the user asks for the current date or time, call get_current_time.\n"
    )
)

Because the `run_agent` function references the agent object we just updated, we can query the agent and it will first reflect and then act

In [11]:
await run_agent("What's the date and time today in Dublin?")

'Today in Dublin, the date and time is **Sunday, October 4, 2026 at 3:21 PM** (EST timezone). Daylight saving time is currently in effect.'

In [12]:
await run_agent("What's the time in Tokio?")

'The current time in Tokyo is **Sunday, October 4, 2026 at 11:21 PM** (23:21) Japan Standard Time (JST).'

## Exercises for the Reader

- Add your own MCP sever.
- Use another existing MCP Server, https://mcpservers.org/

## Conclusions

In this notebook, we created a Pydantic AI agent connected to a local LLM through Ollama. We first showed that, without tools, the agent cannot answer time-dependent questions because its knowledge is limited to its training data.

By connecting an MCP time server to the agent, we enabled it to call external tools at inference time. This allowed the agent to retrieve real-time information such as the current date and time in different time zones.

## References

<div class="alert alert-block alert-info">
<ul>
  <li><a href="https://modelcontextprotocol.io/docs/getting-started/intro">Model Context Protocol</a></li>
</ul>
</div>

---

[AMD University Program](https://www.amd.com/aup).

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.

SPDX-License-Identifier: MIT